# ⚡ DQ Framework — Controller Notebook

**This is the only notebook you need to run.**
It `%run`s the three helper notebooks (config validator, rule engine,
results writer) and then orchestrates the full DQ pipeline.

---

## How it works

| Phase | What happens |
|---|---|
| **0** | Install dependencies (`pyyaml`) |
| **1** | Read widgets → set runtime parameters |
| **2** | `%run` helper notebooks (config validator, rule engine, dataset loader, results writer) |
| **3** | **Config pre-flight** — validate master_config + all dataset configs, abort on ANY error |
| **4** | Load each dataset as a Spark DataFrame |
| **5** | Run all DQ rules, collect results |
| **6** | Write results to Delta tables |
| **7** | Print summary, fail notebook if CRITICAL rules failed |

---

## Widget parameters

| Parameter | Default | Description |
|---|---|---|
| `repo_root` | `/Workspace/Repos/your-name/dq-framework-notebooks` | Full path to the repo root |
| `master_config_path` | `config/master_config.yml` | Relative path from repo_root |
| `dataset_filter` | *(empty = all)* | Comma-separated subset of dataset names |
| `dry_run` | `false` | Validate configs only — do not run rules |
| `abort_on_critical` | `true` | Fail notebook when a CRITICAL rule fails |
| `results_table` | `main.dq_framework.dq_results` | Override global setting |
| `alerts_table` | `main.dq_framework.dq_alerts` | Override global setting |

---

## Path setup for different storage options

**Git Repo (recommended):**
```
repo_root = /Workspace/Repos/your-name/dq-framework-notebooks
```
YAMLs live at e.g. `/Workspace/Repos/your-name/dq-framework-notebooks/config/master_config.yml`

**DBFS / Volumes:**
```
repo_root = /dbfs/FileStore/dq-framework
```
YAMLs live at e.g. `/dbfs/FileStore/dq-framework/config/master_config.yml`
Upload YAMLs via `dbutils.fs.put()` or the Databricks Files UI.

## Step 0 — Install dependencies

In [0]:
%pip install pyyaml --quiet

## Step 1 — Create and read widgets

In [0]:
# Remove existing widgets to avoid stale values on re-run
for widget_name in [
    "repo_root", "master_config_path", "dataset_filter",
    "dry_run", "abort_on_critical", "results_table", "alerts_table"
]:
    try:
        dbutils.widgets.remove(widget_name)
    except Exception:
        pass

# Create widgets
dbutils.widgets.text(
    "repo_root",
    "/Workspace/Users/XXX/mdd-dq-framework",
    "Repo Root (full path)",
)
dbutils.widgets.text(
    "master_config_path",
    "config/master_config.yml",
    "Master Config (relative to repo_root)",
)
dbutils.widgets.text(
    "dataset_filter",
    "",
    "Dataset Filter (comma-sep, empty = all)",
)
dbutils.widgets.dropdown(
    "dry_run",
    "false",
    ["true", "false"],
    "Dry Run (config validation only)",
)
dbutils.widgets.dropdown(
    "abort_on_critical",
    "true",
    ["true", "false"],
    "Abort on CRITICAL failure",
)
dbutils.widgets.text(
    "results_table",
    "wdfe.dq_mgmt.dq_results",
    "Results Delta Table (overrides master config)",
)
dbutils.widgets.text(
    "alerts_table",
    "wdfe.dq_mgmt.dq_alerts",
    "Alerts Delta Table (overrides master config)",
)

In [0]:
# Read widget values
REPO_ROOT           = dbutils.widgets.get("repo_root").strip().rstrip("/")
MASTER_CONFIG_REL   = dbutils.widgets.get("master_config_path").strip()
DATASET_FILTER_RAW  = dbutils.widgets.get("dataset_filter").strip()
DRY_RUN             = dbutils.widgets.get("dry_run").lower() == "true"
ABORT_ON_CRITICAL   = dbutils.widgets.get("abort_on_critical").lower() == "true"
RESULTS_TABLE_PARAM = dbutils.widgets.get("results_table").strip()
ALERTS_TABLE_PARAM  = dbutils.widgets.get("alerts_table").strip()

MASTER_CONFIG_PATH  = f"{REPO_ROOT}/{MASTER_CONFIG_REL}"
DATASET_FILTER      = [x.strip() for x in DATASET_FILTER_RAW.split(",") if x.strip()]

print("=" * 60)
print("  DQ Framework — Runtime Configuration")
print("=" * 60)
print(f"  Repo root         : {REPO_ROOT}")
print(f"  Master config     : {MASTER_CONFIG_PATH}")
print(f"  Dataset filter    : {DATASET_FILTER or 'ALL'}")
print(f"  Dry run           : {DRY_RUN}")
print(f"  Abort on critical : {ABORT_ON_CRITICAL}")
print(f"  Results table     : {RESULTS_TABLE_PARAM}")
print(f"  Alerts table      : {ALERTS_TABLE_PARAM}")
print("=" * 60)

## Step 2 — Load helper notebooks via %run

> **Note:** The `%run` paths below are relative to *this notebook's location*.
> If you move the controller notebook, adjust these paths accordingly.
> All four helper notebooks must be in the same folder.

In [0]:
%run ./dq_config_validator

In [0]:
%run ./dq_rule_engine

In [0]:
%run ./dq_dataset_loader

In [0]:
%run ./dq_results_writer

In [0]:
print("✅ All helper notebooks loaded successfully.")
print(f"   Rule types available : {len(VALID_RULE_TYPES)}")
print(f"   Severities           : {VALID_SEVERITIES}")

## Step 3 — CONFIG PRE-FLIGHT (runs before touching any data)

In [0]:
print("\n" + "=" * 72)
print("  PHASE 1: CONFIG PRE-FLIGHT VALIDATION")
print("=" * 72)

# This is the critical gate. Any problem → DQConfigError → notebook fails cleanly.
try:
    dataset_configs, global_settings = load_all_configs(
        master_path = MASTER_CONFIG_PATH,
        base_dir    = REPO_ROOT,
    )
except DQConfigError as cfg_err:
    # ── Print the full error clearly and abort ────────────────────────────────────────
    border = "🔴 " * 25
    print(f"\n{border}")
    print("  CONFIG VALIDATION FAILED — RUN ABORTED")
    print(f"{border}")
    print(str(cfg_err))
    print(f"{border}\n")
    # Exit the notebook — the job run will show as FAILED with this message
    dbutils.notebook.exit(f"CONFIG_VALIDATION_FAILED: {str(cfg_err)[:800]}")

In [0]:
# ── Apply dataset_filter if specified ─────────────────────────────────────────────
if DATASET_FILTER:
    available_names = {c["dataset"]["name"] for c in dataset_configs}
    invalid_filters = [n for n in DATASET_FILTER if n not in available_names]
    if invalid_filters:
        msg = (
            f"DATASET FILTER ERROR: The following name(s) in dataset_filter are not "
            f"in master_config.yml or are disabled:\n  {invalid_filters}\n"
            f"Available enabled datasets: {sorted(available_names)}"
        )
        print(f"\n🔴 {msg}")
        dbutils.notebook.exit(f"FILTER_ERROR: {msg}")

    dataset_configs = [
        c for c in dataset_configs if c["dataset"]["name"] in DATASET_FILTER
    ]
    print(f"\n[FILTER] Running subset: {[c['dataset']['name'] for c in dataset_configs]}")

print(f"\n✅ Config validation passed. {len(dataset_configs)} dataset(s) ready to validate.")

In [0]:
# ── Dry run ends here ──────────────────────────────────────────────────────────
if DRY_RUN:
    print("\n[DRY RUN] Config validation complete. Rule execution skipped.")
    dbutils.notebook.exit("DRY_RUN_OK: All configs are valid. No rules were executed.")

## Step 4 — Initialise Results Writer

In [0]:
print("\n" + "=" * 72)
print("  PHASE 2: INITIALISE RESULTS TABLES")
print("=" * 72)

# Widget values take priority; fall back to global_settings from master_config.yml
results_table = (
    RESULTS_TABLE_PARAM
    or global_settings.get("results_table", "main.dq_framework.dq_results")
)
alerts_table = (
    ALERTS_TABLE_PARAM
    or global_settings.get("alerts_table", "main.dq_framework.dq_alerts")
)

# Respect widget-level override of abort_on_critical
abort_on_critical = (
    ABORT_ON_CRITICAL
    if ABORT_ON_CRITICAL is not None
    else global_settings.get("abort_on_critical", True)
)

writer = ResultsWriter(spark, results_table, alerts_table)

print(f"  Results table     : {results_table}")
print(f"  Alerts table      : {alerts_table}")
print(f"  Run ID            : {writer.run_id}")
print(f"  Abort on critical : {abort_on_critical}")

## Step 5 — Load Datasets and Execute DQ Rules

In [0]:
print("\n" + "=" * 72)
print("  PHASE 3: EXECUTE DQ RULES")
print("=" * 72)

engine      = RuleEngine(spark)
all_results = []

for dataset_config in dataset_configs:
    ds_meta = dataset_config["dataset"]
    ds_name = ds_meta["name"]
    rules   = dataset_config.get("rules", [])
    source  = ds_meta.get("source", {})

    print(f"\n{'─' * 60}")
    print(f"  Dataset : {ds_name}")
    print(f"  Source  : {source.get('type', '?')} → {source.get('path', '?')}")
    print(f"  Rules   : {len(rules)}")
    print(f"{'─' * 60}")

    # ── Load the DataFrame ──────────────────────────────────────────────────
    try:
        df            = load_dataset(spark, dataset_config)
        total_rows    = df.count()           # cache so rules share the same scan
        print(f"  ✅ Loaded — {total_rows:,} rows")
    except DatasetLoaderError as load_err:
        err_msg = str(load_err)
        print(f"  ❌ LOAD FAILED: {err_msg}")
        # Create a synthetic CRITICAL result representing the load failure
        all_results.append(DQRuleResult(
            dataset_name = ds_name,
            rule_id      = f"{ds_name}.__load__",
            rule_type    = "dataset_load",
            description  = "Dataset could not be loaded from source",
            severity     = "CRITICAL",
            passed       = False,
            error        = err_msg,
        ))
        if abort_on_critical:
            writer.write(all_results)
            writer.print_summary(all_results)
            dbutils.notebook.exit(
                f"CRITICAL_LOAD_FAILURE: Dataset '{ds_name}' could not be loaded. "
                f"Results written to {results_table}."
            )
        continue  # skip to next dataset

    # ── Execute each rule ───────────────────────────────────────────────────
    ds_results = []
    for rule in rules:
        result = engine.execute(rule, df, ds_name)
        ds_results.append(result)

        icon = (
            "✅" if result.passed
            else ("🔴" if result.severity == "CRITICAL" else
                  ("⚠️ " if result.severity == "WARNING" else "ℹ️ "))
        )
        detail_preview = (result.details or result.error or "")[:80]
        print(
            f"  {icon} [{result.severity:<8}] "
            f"{result.rule_id:<40} {result.status}   {detail_preview}"
        )

    all_results.extend(ds_results)


    # ── Per-dataset CRITICAL abort check ─────────────────────────────────────
    ds_critical_fails = [r for r in ds_results if not r.passed and r.severity == "CRITICAL"]
    if ds_critical_fails and abort_on_critical:
        writer.write(all_results)
        writer.print_summary(all_results)
        crit_ids = [r.rule_id for r in ds_critical_fails]
        abort_msg = (
            f"CRITICAL DQ FAILURE on dataset '{ds_name}'. Aborting (abort_on_critical=true).\n"
            f"Failed rules : {crit_ids}\n"
            f"Results at   : {results_table}"
        )
        print(f"\n{'🔴 ' * 25}")
        print(f"  {abort_msg}")
        print(f"{'🔴 ' * 25}\n")
        dbutils.notebook.exit(f"CRITICAL_DQ_FAILURE: {abort_msg}")

## Step 6 — Persist Results to Delta

In [0]:
print("\n" + "=" * 72)
print("  PHASE 4: PERSIST RESULTS")
print("=" * 72)

writer.write(all_results)
print(f"  ✅ {len(all_results)} result(s) written to {results_table}")

# Show results inline as a Spark DataFrame for easy inspection
print("\n  Inline results preview (latest run):")
spark.table(results_table) \
    .filter(f"run_id = '{writer.run_id}'") \
    .select("dataset_name", "rule_id", "severity", "status", "row_count", "details") \
    .display()

## Step 7 — Print Summary and Final Exit

In [0]:
writer.print_summary(all_results)

In [0]:
critical_failures = [r for r in all_results if not r.passed and r.severity == "CRITICAL"]
warning_failures  = [r for r in all_results if not r.passed and r.severity == "WARNING"]

if critical_failures:
    crit_ids = [f"{r.dataset_name}.{r.rule_id}" for r in critical_failures]
    msg = (
        f"DQ RUN COMPLETED WITH {len(critical_failures)} CRITICAL FAILURE(S).\n"
        f"Failed rules : {crit_ids}\n"
        f"Warnings     : {len(warning_failures)}\n"
        f"Results at   : {results_table}\n"
        f"Run ID       : {writer.run_id}"
    )
    print("🔴 " + msg)
    # Raise exception → Databricks marks the job run as FAILED
    raise Exception(f"CRITICAL_DQ_FAILURE: {msg}")

elif warning_failures:
    warn_ids = [f"{r.dataset_name}.{r.rule_id}" for r in warning_failures]
    summary_msg = (
        f"WARNINGS_ONLY: {len(warning_failures)} warning rule(s) failed. "
        f"No CRITICAL failures. "
        f"Results: {results_table}. Run ID: {writer.run_id}"
    )
    print(f"⚠️  {summary_msg}")
    dbutils.notebook.exit(summary_msg)

else:
    success_msg = (
        f"ALL_PASSED: {len(all_results)} rule(s) passed across "
        f"{len(dataset_configs)} dataset(s). "
        f"Run ID: {writer.run_id}"
    )
    print(f"✅ {success_msg}")
    dbutils.notebook.exit(success_msg)